**Task**  
Estimate shrub confidence scores across different locations (e.g. center of shrub to edge of mask).  
  
**Plan**  
Use Shrub center as multivariate mean.  
Calculate covariance matrix.  
Estimate shrub confidence at all shrub pixels (center of pixel for simplicity)  

**Later Revisions**  
It would be ideal to get a better mean and covariance estimates by averaging over more shrubs (maybe divide by arbitrary bounding boxes for calculating this)  
Without this estimation a normal distribution is fairly arbitrary since LLN won't be convergent for one shrub.

**Plan 2**  
Estimate shrub islands by their connected-ness. More connected shrubs are more likely to actually part of a shrub. A normalization of this island kernel can be multiplied by the normal kernel for a better estimate.

In [ ]:
import numpy as np

grid_mask = np.array([
    [0,0,0,1,0],
    [0,0,1,1,0],
    [0,1,0,0,0],
    [0,1,1,1,0]
])

In [6]:
np.pad(grid_mask, 1)

array([[0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 1, 1, 0, 0],
       [0, 0, 1, 0, 0, 0, 0],
       [0, 0, 1, 1, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0]])

**For further improvement**  
Additional functionality needs to be added to keep track of the original coordinates relative to grid where the shrub spanned. This is what bounding_box contains, but work needs to be done to return a dictionary instead as other information may be required for other processes.

In [ ]:
from scipy.ndimage import label, find_objects
from typing import List

def get_shrub_list(grid: np.ndarray) --> List:
    island_kernel = np.array([
        [0,1,0],
        [1,1,1],
        [0,1,0]
    ])

    labelled_grid, _ = label(grid, structure=island_kernel)
    slices = find_objects(labelled_grid)

    shrub_list = list()
    for group_id, bounding_box in enumerate(slices, start=1):
        individual_shrub = grid_mask[bounding_box]
        group_mask = labelled_grid[bounding_box] == group_id
        individual_shrub = np.where(group_mask, individual_shrub, 0)
        shrub_list.append(individual_shrub)

    return shrub_list

get_shrub_list(grid_mask)

[array([[0, 1],
        [1, 1]]),
 array([[1, 0, 0],
        [1, 1, 1]])]

**Idea**  
The island kernel produces islands (i.e. immediately adjacent groups of pixels).  
I can estimate the Euclidean distance, but this doesn't keep the property of kernels summing to 0 (i.e. producing no residual).
Since a distance of 0 is high association I give the center of the kernel a value of 1. 
Since I produced this shrub mask with the island kernel I know that there will be adjacent neighbors, so I weight those positively.
Since I know the current shrub has no additional adjacent neighbors the most likely neighbors will be diagonal. If there is an adjacent tile that was not included it is likely that the tile includes a small part of the shrub.  
  
Another path to consider is relative kernel likelihoods. I performed transforms with fairly arbitrary multiples, emphasizing shape. In this way I have a universal scale to compare shrubs that could never be in the same region/directly comparable. Alternatively, you can scale according to the group by either a) forming kernels that are depending on the individual shrub characteristics before processing or b) form kernels that have a sum of 1 where you instead consider the expected weight disbursement of the shrub in the nearby region. The latter of course can fall victim to disconnected shrubs which are infeasible, making an effective solution with that more difficult.  
  
**Drawbacks (global implementation)**  
By acknowledging that the most likely reason shrubs are missing is because they are diagonal, I give extra, potentially too much, weighting for this optimistic perspective. While diagonals should have considerable weighting, what a good balance for this waiting is is a bit artsy and will require large scale testing that would be easier after full pipeline implementation.

In [ ]:
from scipy.signal import convolve2d

def get_shrub_conf_universal(shrub: np.ndarray, mean: int|np.ndarray=None, var: int|np.ndarray=None, eps: float=np.e) -> np.ndarray:
    """
    Applies a 1-padded convolution to estimate confidence of shrub existence around an individual shrub

    Args:
        shrub - M x N array containing a single shrub (as defined by the island kernel)
        mean - precalculated mean of appropriate dimension
        var - precalculated variance of appropriate dimension
        eps - increase epsilon to reducing diagonal edge weighting

    Returns:
        weighted_shrub - (M+1) x (N+1) array containing confidence estimates in (0,1]
    """
    transform_diag = -np.sqrt(1/(2+eps**2))
    transform_adj = 1/(1+eps)
    transform_kernel = (1/np.sum(np.abs([1, 4*transform_adj, 4*transform_diag]))) * np.array([
        [transform_diag, transform_adj, transform_diag],
        [transform_adj,1,transform_adj],
        [transform_diag, transform_adj, transform_diag],
    ])

    weighted_shrub = convolve2d(np.pad(shrub, 1), transform_kernel, mode="same") 
    if mean is None:
        mean = weighted_shrub.mean()

    if var is None:
        var = weighted_shrub.var()

    weighted_shrub = 1 - np.e**(-(weighted_shrub-mean)**2/(2*var)) # standard normal weighting. denser shrub regions will produce a greater contrast

    return weighted_shrub

def get_weighted_shrubs(grid: np.ndarray) -> np.ndarray:
    """
    Aggregator function for finding and weighting shrubs from arbitrary tiling

    Args:
        grid - M x N binary array with shrubs as 1 and non-shrub as 0
    
    Returns:
        weighted_shrubs - (M+1) x (N+1) array with confidence values in (0,1]
    """
    shrub_list = get_shrub_list(grid)
    weighted_shrubs = list()
    for shrub in shrub_list:
        weighted_shrub = get_shrub_conf_universal(shrub)
        weighted_shrubs.append(weighted_shrub)

    return weighted_shrubs

weighted_shrubs = get_weighted_shrubs(grid_mask)
weighted_shrubs

[array([[0.0360695 , 0.32313353, 0.02692443, 0.32313353],
        [0.32313353, 0.00789383, 0.67427749, 0.06923091],
        [0.02692443, 0.67427749, 0.96725321, 0.06923091],
        [0.32313353, 0.06923091, 0.06923091, 0.32313353]]),
 array([[0.29101859, 0.01947053, 0.29101859, 0.03475306, 0.03475306],
        [0.06447233, 0.60405693, 0.10214004, 0.06447233, 0.29101859],
        [0.06447233, 0.94235497, 0.8110475 , 0.84309329, 0.01947053],
        [0.29101859, 0.06447233, 0.35016631, 0.06447233, 0.29101859]])]

In [3]:
import numpy as np
from scipy.ndimage import label, find_objects
from scipy.signal import convolve2d
from typing import List

def get_shrub_list(grid: np.ndarray) -> List:
    island_kernel = np.array([
        [0,1,0],
        [1,1,1],
        [0,1,0]
    ])

    labelled_grid, _ = label(grid, structure=island_kernel)
    slices = find_objects(labelled_grid)

    shrub_list = list()
    for group_id, bounding_box in enumerate(slices, start=1):
        individual_shrub = grid[bounding_box]
        group_mask = labelled_grid[bounding_box] == group_id
        individual_shrub = np.where(group_mask, individual_shrub, 0)
        shrub_list.append(individual_shrub)

    return shrub_list

def get_shrub_conf_universal(shrub: np.ndarray, mean: int|np.ndarray=None, var: int|np.ndarray=None, eps: float=np.e) -> np.ndarray:
    """
    Applies a 1-padded convolution to estimate confidence of shrub existence around an individual shrub

    Args:
        shrub - M x N array containing a single shrub (as defined by the island kernel)
        mean - precalculated mean of appropriate dimension
        var - precalculated variance of appropriate dimension
        eps - increase epsilon to reducing diagonal edge weighting

    Returns:
        weighted_shrub - (M+1) x (N+1) array containing confidence estimates in (0,1]
    """
    transform_diag = -np.sqrt(1/(2+eps**2))
    transform_adj = 1/(1+eps)
    transform_kernel = (1/np.sum(np.abs([1, 4*transform_adj, 4*transform_diag]))) * np.array([
        [transform_diag, transform_adj, transform_diag],
        [transform_adj,1,transform_adj],
        [transform_diag, transform_adj, transform_diag],
    ])

    weighted_shrub = convolve2d(np.pad(shrub, 1), transform_kernel, mode="same") 
    if mean is None:
        mean = weighted_shrub.mean()

    if var is None:
        var = weighted_shrub.var()

    weighted_shrub = 1 - np.e**(-(weighted_shrub-mean)**2/(2*var)) # standard normal weighting. denser shrub regions will produce a greater contrast

    return weighted_shrub

def get_weighted_shrubs(grid: np.ndarray) -> np.ndarray:
    """
    Aggregator function for finding and weighting shrubs from arbitrary tiling

    Args:
        grid - M x N binary array with shrubs as 1 and non-shrub as 0
    
    Returns:
        weighted_shrubs - (M+1) x (N+1) array with confidence values in (0,1]
    """
    shrub_list = get_shrub_list(grid)
    weighted_shrubs = list()
    for shrub in shrub_list:
        weighted_shrub = get_shrub_conf_universal(shrub)
        weighted_shrubs.append(weighted_shrub)

    return weighted_shrubs


grid_mask = np.array([
    [0,0,0,1,0],
    [0,0,1,1,0],
    [0,1,0,0,0],
    [0,1,1,1,0]
])
weighted_shrubs = get_weighted_shrubs(grid_mask)
weighted_shrubs

[array([[0.0360695 , 0.32313353, 0.02692443, 0.32313353],
        [0.32313353, 0.00789383, 0.67427749, 0.06923091],
        [0.02692443, 0.67427749, 0.96725321, 0.06923091],
        [0.32313353, 0.06923091, 0.06923091, 0.32313353]]),
 array([[0.29101859, 0.01947053, 0.29101859, 0.03475306, 0.03475306],
        [0.06447233, 0.60405693, 0.10214004, 0.06447233, 0.29101859],
        [0.06447233, 0.94235497, 0.8110475 , 0.84309329, 0.01947053],
        [0.29101859, 0.06447233, 0.35016631, 0.06447233, 0.29101859]])]